In [1]:
import pandas as pd

data = [
    # device A
    ["A", "2026-06-29 08:00:00", 70, "NORMAL"],
    ["A", "2026-06-29 08:01:00", 86, "HIGH"],
    ["A", "2026-06-29 08:02:00", 88, "HIGH"],
    ["A", "2026-06-29 08:03:00", 72, "NORMAL"],
    ["A", "2026-06-29 08:04:00", 91, "HIGH"],
    ["A", "2026-06-29 08:05:00", 93, "HIGH"],
    ["A", "2026-06-29 08:06:00", 95, "HIGH"],

    # device B
    ["B", "2026-06-29 08:00:00", 80, "NORMAL"],
    ["B", "2026-06-29 08:01:00", 89, "HIGH"],
    ["B", "2026-06-29 08:02:00", 76, "NORMAL"],
    ["B", "2026-06-29 08:03:00", 90, "HIGH"],
    ["B", "2026-06-29 08:04:00", 92, "HIGH"],
]

df = pd.DataFrame(
    data,
    columns=[
        "device_id",
        "collect_time",
        "temp_value",
        "status"
    ]
)

df["collect_time"] = pd.to_datetime(df["collect_time"])

print(df)

# 保存为 csv（可选）
df.to_csv("sensor_log.csv", index=False)

   device_id        collect_time  temp_value  status
0          A 2026-06-29 08:00:00          70  NORMAL
1          A 2026-06-29 08:01:00          86    HIGH
2          A 2026-06-29 08:02:00          88    HIGH
3          A 2026-06-29 08:03:00          72  NORMAL
4          A 2026-06-29 08:04:00          91    HIGH
5          A 2026-06-29 08:05:00          93    HIGH
6          A 2026-06-29 08:06:00          95    HIGH
7          B 2026-06-29 08:00:00          80  NORMAL
8          B 2026-06-29 08:01:00          89    HIGH
9          B 2026-06-29 08:02:00          76  NORMAL
10         B 2026-06-29 08:03:00          90    HIGH
11         B 2026-06-29 08:04:00          92    HIGH


## 要求

### 分别用 SQL 和 Pandas 完成：

#### 找出每个设备中，连续高温达到 2 次及以上的时间段。

## 异常定义
`temp_value >= 85 OR status = 'HIGH'`

## 最终输出

|device_id|	high_start_time|high_end_time|high_duration_count|
|---------|----------------|-------------|-------------------|
|A|	2026-06-29 08:01:00	|2026-06-29 08:02:00	|2|
|A	|2026-06-29 08:04:00	|2026-06-29 08:06:00	|3|
|B	|2026-06-29 08:03:00	|2026-06-29 08:04:00|	2|

In [45]:
# SQL轨道

import duckdb
query = """
WITH temp_bool AS (
SELECT  
    device_id,
    collect_time,
    temp_value,
    status,
    CASE WHEN temp_value >= 85 OR status = 'HIGH' THEN True ELSE False END AS temp_value_bool
FROM df
),
last_line_temp_bool AS(
SELECT
    device_id,
    collect_time,
    temp_value,
    status,
    temp_value_bool,
    LAG(temp_value_bool)
      OVER(
        PARTITION BY device_id 
            ORDER BY collect_time
        ) AS preview_temp_bool
FROM temp_bool
ORDER BY device_id
),
sign_start AS(
SELECT
    device_id,
    collect_time,
    temp_value,
    status,
    temp_value_bool,
    preview_temp_bool,
    CASE WHEN temp_value_bool = True AND preview_temp_bool = False THEN 1 ELSE 0 END AS high_start
FROM last_line_temp_bool
),
segmented_mark_table AS(
SELECT
    device_id,
    collect_time,
    temp_value,
    status,
    high_start,
    SUM(high_start) OVER(PARTITION BY device_id ORDER BY collect_time)::INTEGER AS segmented_mark
FROM sign_start
),
delete_normal AS(
SELECT
    device_id,
    collect_time,
    temp_value,
    high_start,
    status,
    segmented_mark
FROM segmented_mark_table
WHERE temp_value >= 85 OR status = 'HIGH'
),
high_temp_continue AS (
SELECT
    device_id,
    MIN(collect_time) AS high_start_time,
    MAX(collect_time) AS high_end_time,
    COUNT(*) AS high_duration_count
FROM delete_normal
GROUP BY device_id,segmented_mark
)
SELECT *
FROM high_temp_continue
WHERE high_duration_count >= 2
ORDER BY device_id
"""
# 调整显示设置
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 25)

df_sql = duckdb.execute(query).fetchdf()
print(df_sql)

  device_id     high_start_time       high_end_time  high_duration_count
0         A 2026-06-29 08:01:00 2026-06-29 08:02:00                    2
1         A 2026-06-29 08:04:00 2026-06-29 08:06:00                    3
2         B 2026-06-29 08:03:00 2026-06-29 08:04:00                    2


In [59]:
# python 轨道

df = df.sort_values(['device_id', 'collect_time']).copy()

df['is_high'] = (df['temp_value'] >= 85) | (df['status'] == 'HIGH')

df['start_mark'] = (
    df['is_high'] 
    & (df.groupby('device_id')['is_high'].shift(1).fillna(False)==False)
).astype(int)

df['phase_mark'] = df.groupby('device_id')['start_mark'].cumsum()

df_continue_high = (
    df[df['is_high']]
    .groupby(['device_id', 'phase_mark'], as_index=False)
    .agg(
        high_start_time=('collect_time', 'min'),
        high_end_time=('collect_time', 'max'),
        high_duration_count=('collect_time', 'size')
    )
    .query('high_duration_count >= 2')
    [['device_id', 'high_start_time', 'high_end_time', 'high_duration_count']]
)

print(df_continue_high)

  device_id     high_start_time       high_end_time  high_duration_count
0         A 2026-06-29 08:01:00 2026-06-29 08:02:00                    2
1         A 2026-06-29 08:04:00 2026-06-29 08:06:00                    3
3         B 2026-06-29 08:03:00 2026-06-29 08:04:00                    2
